# Lab 1 — LLM Data Pipeline

## Step 0 — Install & Import

In [ ]:
# Install all dependencies
!pip install -q arxiv requests sentence-transformers chromadb \
    bertopic umap-learn hdbscan plotly pandas numpy scikit-learn \
    nltk tqdm numba
print('✅ All packages installed')

In [ ]:
import os, re, json, time, hashlib, warnings, logging
import numpy as np
import pandas as pd
import requests
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm

import nltk
from nltk.corpus import stopwords

import arxiv
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

import chromadb

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
np.random.seed(42)

print('✅ Imports done')
print(f'🕒 Run started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## Step 1 — Data Schema & Configuration

In [ ]:
@dataclass
class Article:
    """Unified schema for all data sources."""
    id:             str
    title:          str
    text:           str            # Abstract or body
    source:         str            # 'arxiv' | 'hackernews'
    url:            str
    authors:        List[str]
    published_date: str            # YYYY-MM-DD
    categories:     List[str]
    score:          int  = 0       # HN upvotes (0 for arXiv)
    word_count:     int  = 0       # computed after cleaning
    quality_score:  float = 0.0   # computed quality metric
    chunks:         List[str] = field(default_factory=list)


# ── Pipeline configuration ──────────────────────────────────────────────────
CFG = {
    # Collection
    "arxiv_queries": [
        "MLOps machine learning operations production",
        "LLMOps large language model deployment fine-tuning",
        "vector database retrieval augmented generation",
        "transformer NLP text classification benchmark",
        "data pipeline orchestration feature engineering",
    ],
    "arxiv_per_query":  12,
    "hn_per_query":     10,
    "hn_min_score":     30,

    # Preprocessing
    "min_words":        40,
    "chunk_size":       150,   # words
    "chunk_overlap":    25,

    # Embeddings
    "embed_model":      "all-MiniLM-L6-v2",
    "embed_batch":      32,

    # BERTopic
    "n_topics":         8,
    "min_topic_size":   3,

    # ChromaDB
    "chroma_path":      "./chroma_lab1",
    "collection":       "llm_pipeline_lab1",
}

print('✅ Schema and config ready')
print(f'   arXiv queries : {len(CFG["arxiv_queries"])}')
print(f'   Embedding model: {CFG["embed_model"]}')
print(f'   Chunk size    : {CFG["chunk_size"]} words  (overlap {CFG["chunk_overlap"]})')

## Step 2 — Dual-Source Data Collection

### 2a — arXiv Papers

In [ ]:
def collect_arxiv(queries: List[str], per_query: int) -> List[Article]:
    articles, seen = [], set()
    for q in tqdm(queries, desc='📚 arXiv'):
        try:
            search = arxiv.Search(
                query=q, max_results=per_query,
                sort_by=arxiv.SortCriterion.Relevance
            )
            for r in search.results():
                pid = r.entry_id.split('/')[-1]
                if pid in seen:
                    continue
                seen.add(pid)
                articles.append(Article(
                    id=pid,
                    title=r.title.strip(),
                    text=r.summary.strip(),
                    source='arxiv',
                    url=r.entry_id,
                    authors=[a.name for a in r.authors[:5]],
                    published_date=r.published.strftime('%Y-%m-%d'),
                    categories=list(r.categories),
                ))
            time.sleep(0.4)
        except Exception as e:
            print(f'  ⚠️  arXiv query failed: {q[:40]} — {e}')
    return articles


arxiv_arts = collect_arxiv(CFG['arxiv_queries'], CFG['arxiv_per_query'])
print(f'\n✅ arXiv: {len(arxiv_arts)} papers collected')
print(f'   Date range : {min(a.published_date for a in arxiv_arts)} → '
      f'{max(a.published_date for a in arxiv_arts)}')
print(f'   Sample     : {arxiv_arts[0].title[:80]}...')

### 2b — HackerNews Stories

In [ ]:
def collect_hackernews(queries: List[str], per_query: int, min_score: int) -> List[Article]:
    articles, seen = [], set()
    for q in tqdm(queries, desc='🔥 HN'):
        try:
            resp = requests.get(
                'https://hn.algolia.com/api/v1/search',
                params={'query': q, 'tags': 'story',
                        'hitsPerPage': per_query,
                        'numericFilters': f'points>={min_score}'},
                timeout=8
            )
            for hit in resp.json().get('hits', []):
                oid = str(hit.get('objectID', ''))
                if oid in seen:
                    continue
                seen.add(oid)
                body = hit.get('story_text') or hit.get('title', '')
                body = re.sub(r'<[^>]+>', ' ', body)   # strip HTML
                body = re.sub(r'\s+', ' ', body).strip()
                articles.append(Article(
                    id=oid,
                    title=hit.get('title', '').strip(),
                    text=body or hit.get('title', ''),
                    source='hackernews',
                    url=hit.get('url') or f'https://news.ycombinator.com/item?id={oid}',
                    authors=[hit.get('author', 'unknown')],
                    published_date=(hit.get('created_at', '2024-01-01')[:10]),
                    categories=['hackernews', q.replace(' ', '_')],
                    score=hit.get('points', 0),
                ))
            time.sleep(0.3)
        except Exception as e:
            print(f'  ⚠️  HN query failed: {q} — {e}')
    return articles


HN_QUERIES = ['machine learning', 'LLM deployment', 'vector database', 'MLOps', 'AI pipeline']
hn_arts = collect_hackernews(HN_QUERIES, CFG['hn_per_query'], CFG['hn_min_score'])

all_articles = arxiv_arts + hn_arts
print(f'\n✅ HackerNews: {len(hn_arts)} stories collected')
print(f'\n🎯 TOTAL CORPUS: {len(all_articles)} articles '
      f'({len(arxiv_arts)} arXiv + {len(hn_arts)} HN)')

## Step 3 — Text Preprocessing

In [ ]:
def clean_text(text: str) -> str:
    """Remove LaTeX, HTML, URLs, normalise whitespace."""
    text = re.sub(r'\$[^$]+\$', ' ', text)           # LaTeX math
    text = re.sub(r'<[^>]+>', ' ', text)              # HTML tags
    text = re.sub(r'http\S+', ' ', text)              # URLs
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:\-]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def quality_score(a: Article) -> float:
    """Score 0-1 combining word-count, source, and social signal."""
    wc = a.word_count
    s = 0.0
    # Word count (ideal 80-600)
    if 80 <= wc <= 600:    s += 0.35
    elif 40 <= wc < 80:    s += 0.15
    elif wc > 600:         s += 0.20
    # Source
    if a.source == 'arxiv':        s += 0.35
    elif a.source == 'hackernews': s += 0.10 + min(a.score / 1000, 0.20)
    # Title length
    tw = len(a.title.split())
    if 5 <= tw <= 20: s += 0.15
    elif tw > 0:      s += 0.05
    # Author signal
    s += 0.10 if len(a.authors) >= 2 else 0.05
    return min(round(s, 3), 1.0)


def chunk_text(text: str, size: int, overlap: int) -> List[str]:
    """Split into overlapping word-window chunks."""
    words = text.split()
    if len(words) <= size:
        return [text]
    step = size - overlap
    return [
        ' '.join(words[i:i+size])
        for i in range(0, len(words), step)
        if len(words[i:i+size]) >= 15
    ]


def preprocess(articles: List[Article]) -> Tuple[List[Article], dict]:
    stats = dict(input=len(articles), dropped_empty=0, dropped_short=0, chunks=0)
    out = []
    for a in tqdm(articles, desc='🧹 Preprocessing'):
        a.text  = clean_text(a.text)
        a.title = clean_text(a.title)
        if not a.text:
            stats['dropped_empty'] += 1; continue
        a.word_count = len(a.text.split())
        if a.word_count < CFG['min_words']:
            stats['dropped_short'] += 1; continue
        a.quality_score = quality_score(a)
        a.chunks = chunk_text(a.title + '. ' + a.text,
                              CFG['chunk_size'], CFG['chunk_overlap'])
        stats['chunks'] += len(a.chunks)
        out.append(a)
    stats.update(output=len(out),
                 retention=round(len(out)/len(articles)*100, 1),
                 avg_chunks=round(stats['chunks']/max(len(out),1), 2))
    return out, stats


clean_articles, prep_stats = preprocess(all_articles)

print('\n📊 Preprocessing Report')
for k, v in prep_stats.items():
    print(f'   {k:<18}: {v}')

## Step 4 — Sentence Embeddings

In [ ]:
print(f'Loading {CFG["embed_model"]}...')
embedder = SentenceTransformer(CFG['embed_model'])
print(f'✅ Loaded  |  dim = {embedder.get_sentence_embedding_dimension()}')


def embed_articles(articles: List[Article]):
    """Flatten all chunks → embed → return ids, vectors, metadata, texts."""
    ids, texts, metas = [], [], []
    for a in articles:
        for ci, chunk in enumerate(a.chunks):
            ids.append(f"{a.id}_c{ci}")
            texts.append(chunk)
            metas.append({
                'article_id':    a.id,
                'title':         a.title[:200],
                'source':        a.source,
                'url':           a.url[:400],
                'date':          a.published_date,
                'categories':    ', '.join(a.categories[:5]),
                'quality_score': float(a.quality_score),
                'hn_score':      int(a.score),
                'word_count':    int(a.word_count),
                'chunk_index':   ci,
                'total_chunks':  len(a.chunks),
            })

    print(f'Embedding {len(texts)} chunks in batches of {CFG["embed_batch"]}...')
    vectors = embedder.encode(
        texts,
        batch_size=CFG['embed_batch'],
        normalize_embeddings=True,
        show_progress_bar=True
    )
    print(f'✅ Embeddings shape: {vectors.shape}')
    return ids, vectors, metas, texts


chunk_ids, embeddings, chunk_meta, chunk_texts = embed_articles(clean_articles)
print(f'   Memory: {embeddings.nbytes / 1024**2:.2f} MB')

## Step 5 — BERTopic: Neural Topic Modeling

In [ ]:
# Embed article-level texts (not chunks) for cleaner topic discovery
article_texts = [a.title + '. ' + a.text for a in clean_articles]
print(f'Embedding {len(article_texts)} article-level texts for BERTopic...')
art_vectors = embedder.encode(
    article_texts, batch_size=CFG['embed_batch'],
    normalize_embeddings=True, show_progress_bar=True
)

vectorizer = CountVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2))
topic_model = BERTopic(
    vectorizer_model=vectorizer,
    nr_topics=CFG['n_topics'],
    min_topic_size=CFG['min_topic_size'],
    calculate_probabilities=True,
    verbose=False
)

topics, probs = topic_model.fit_transform(article_texts, art_vectors)

topic_info = topic_model.get_topic_info()
n_real = len(topic_info[topic_info.Topic != -1])
n_outlier = sum(1 for t in topics if t == -1)

print(f'\n✅ BERTopic done')
print(f'   Topics discovered : {n_real}')
print(f'   Outlier docs (-1) : {n_outlier}')
print()

for _, row in topic_info[topic_info.Topic != -1].head(8).iterrows():
    words = topic_model.get_topic(row.Topic)
    kw = ', '.join(w for w, _ in words[:5]) if words else '—'
    print(f"   Topic {row.Topic:2d} | {row.Count:3d} docs | {kw}")

# Attach topic labels to articles
for i, a in enumerate(clean_articles):
    if i < len(topics):
        a.categories.append(f'topic_{topics[i]}')

## Step 6 — ChromaDB Vector Store

In [ ]:
import shutil
# Fresh run — delete old DB if exists
if os.path.exists(CFG['chroma_path']):
    shutil.rmtree(CFG['chroma_path'])

chroma_client = chromadb.PersistentClient(path=CFG['chroma_path'])
collection = chroma_client.create_collection(
    name=CFG['collection'],
    metadata={'hnsw:space': 'cosine'}
)

# Upsert in batches of 100
BATCH = 100
for i in tqdm(range(0, len(chunk_ids), BATCH), desc='💾 ChromaDB'):
    sl = slice(i, i + BATCH)
    clean_m = [
        {k: (str(v) if not isinstance(v, (str, int, float, bool)) else v)
         for k, v in m.items()}
        for m in chunk_meta[sl]
    ]
    collection.add(
        ids=chunk_ids[sl],
        embeddings=embeddings[sl].tolist(),
        documents=chunk_texts[sl],
        metadatas=clean_m,
    )

print(f'\n✅ ChromaDB stored {collection.count()} chunks')
print(f'   Path: {CFG["chroma_path"]}')

## Step 7 — Semantic Search Demo

In [ ]:
def semantic_search(query: str, n: int = 5, source: str = None):
    """
    Run cosine-similarity search against ChromaDB.
    Optionally filter by source ('arxiv' or 'hackernews').
    """
    qvec = embedder.encode([query], normalize_embeddings=True).tolist()
    where = {'source': source} if source else None
    results = collection.query(
        query_embeddings=qvec,
        n_results=n,
        where=where,
        include=['documents', 'metadatas', 'distances']
    )
    print(f'\n🔍 Query : "{query}"' + (f'  [filter: source={source}]' if source else ''))
    print('─' * 70)
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        sim = round(1 - dist, 4)  # cosine distance → similarity
        print(f'  [{meta["source"].upper():12s}] sim={sim:.3f}  quality={meta["quality_score"]:.2f}')
        print(f'  Title  : {meta["title"][:75]}')
        print(f'  Snippet: {doc[:120]}...')
        print()


# Demo queries
semantic_search('how to deploy a machine learning model in production')
semantic_search('fine-tuning large language models efficiently', source='arxiv')
semantic_search('vector database for semantic search', n=3, source='hackernews')

## Step 8 — Interactive Data Quality Dashboard

In [ ]:
# Build summary dataframe
df = pd.DataFrame([
    {
        'source':        a.source,
        'word_count':    a.word_count,
        'quality_score': a.quality_score,
        'hn_score':      a.score,
        'n_chunks':      len(a.chunks),
        'n_authors':     len(a.authors),
        'date':          a.published_date,
        'topic':         next((c for c in a.categories if c.startswith('topic_')), 'topic_-1'),
    }
    for a in clean_articles
])

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year_month'] = df['date'].dt.to_period('M').astype(str)
print(f'DataFrame shape: {df.shape}')
df.head()

In [ ]:
# ── 1. Source breakdown ──────────────────────────────────────────────────────
src_counts = df['source'].value_counts().reset_index()
src_counts.columns = ['source', 'count']

fig1 = px.pie(
    src_counts, names='source', values='count',
    title='📚 Corpus — Source Breakdown',
    color_discrete_map={'arxiv': '#4C78A8', 'hackernews': '#F58518'},
    hole=0.4
)
fig1.update_traces(textposition='inside', textinfo='percent+label')
fig1.show()

In [ ]:
# ── 2. Quality score distribution by source ──────────────────────────────────
fig2 = px.histogram(
    df, x='quality_score', color='source', nbins=25, barmode='overlay',
    opacity=0.75,
    color_discrete_map={'arxiv': '#4C78A8', 'hackernews': '#F58518'},
    title='🏅 Quality Score Distribution by Source',
    labels={'quality_score': 'Quality Score (0-1)', 'count': 'Count'}
)
fig2.add_vline(
    x=df['quality_score'].mean(), line_dash='dash',
    annotation_text=f'mean={df["quality_score"].mean():.2f}',
    annotation_position='top right'
)
fig2.show()

In [ ]:
# ── 3. Word count violin ─────────────────────────────────────────────────────
fig3 = px.violin(
    df, y='word_count', x='source', color='source', box=True, points='all',
    color_discrete_map={'arxiv': '#4C78A8', 'hackernews': '#F58518'},
    title='📝 Word Count Distribution by Source',
    labels={'word_count': 'Words per Article'}
)
fig3.show()

In [ ]:
# ── 4. BERTopic bar ──────────────────────────────────────────────────────────
ti = topic_info[topic_info.Topic != -1].copy()
ti['keywords'] = ti.Topic.apply(
    lambda t: ', '.join(w for w, _ in (topic_model.get_topic(t) or [])[:4])
)
ti['label'] = ti.apply(lambda r: f"Topic {r.Topic}: {r.keywords}", axis=1)

fig4 = px.bar(
    ti.sort_values('Count', ascending=True),
    x='Count', y='label', orientation='h',
    color='Count', color_continuous_scale='Teal',
    title='🧠 BERTopic — Document Count per Topic',
    labels={'Count': 'Documents', 'label': ''}
)
fig4.update_layout(height=max(300, len(ti) * 50), showlegend=False)
fig4.show()

In [ ]:
# ── 5. Quality vs Word-Count scatter ─────────────────────────────────────────
fig5 = px.scatter(
    df, x='word_count', y='quality_score', color='source',
    size='n_chunks', hover_data=['topic'],
    color_discrete_map={'arxiv': '#4C78A8', 'hackernews': '#F58518'},
    title='🔬 Quality Score vs Word Count (size = chunks)',
    labels={'word_count': 'Word Count', 'quality_score': 'Quality Score'}
)
fig5.show()

In [ ]:
# ── 6. Articles over time ────────────────────────────────────────────────────
timeline = (
    df.dropna(subset=['year_month'])
      .groupby(['year_month', 'source'])
      .size().reset_index(name='count')
      .sort_values('year_month')
)

fig6 = px.bar(
    timeline, x='year_month', y='count', color='source', barmode='stack',
    color_discrete_map={'arxiv': '#4C78A8', 'hackernews': '#F58518'},
    title='📅 Article Volume Over Time',
    labels={'year_month': 'Month', 'count': 'Articles'}
)
fig6.update_xaxes(tickangle=45)
fig6.show()

## Step 9 — Pipeline Summary

In [ ]:
summary = {
    'run_timestamp':      datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_articles':     len(clean_articles),
    'arxiv_articles':     sum(1 for a in clean_articles if a.source == 'arxiv'),
    'hn_articles':        sum(1 for a in clean_articles if a.source == 'hackernews'),
    'total_chunks':       prep_stats['chunks'],
    'embedding_model':    CFG['embed_model'],
    'embedding_dim':      embedder.get_sentence_embedding_dimension(),
    'embeddings_shape':   str(embeddings.shape),
    'topics_discovered':  n_real,
    'chroma_docs':        collection.count(),
    'avg_quality_score':  round(df['quality_score'].mean(), 3),
    'avg_word_count':     round(df['word_count'].mean(), 1),
}

print('='*55)
print('  PIPELINE SUMMARY — LLM Data Pipeline Lab 1 (Enhanced)')
print('='*55)
for k, v in summary.items():
    print(f'  {k:<25}: {v}')
print('='*55)

# Save as JSON for logging / CI
with open('pipeline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\n✅ Summary saved to pipeline_summary.json')